# ITS — Controlled Score-Based Sampler

Uses the `src/its.*` API.

## Contents
1. Load combined controlled checkpoint
2. Controlled SDE sampling
3. Visualise samples and compare to baseline
4. Inspect thermodynamic diagnostics (path KL, Jarzynski, entropy)
5. Quick benchmark: controlled vs. DDPM vs. DDIM

In [ ]:
import sys
from pathlib import Path

ROOT = Path('.').resolve().parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
print('Project root:', ROOT)

In [ ]:
import torch
from its.models import ScoreUNetConfig, build_score_model
from its.controllers import ControlConfig, build_control_policy, ConvControlConfig, build_conv_control_policy
from its.sde import ScoreSDEConfig, ScoreSDESimulator

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

## 1. Load controlled checkpoint

In [ ]:
CKPT_DIR = ROOT / 'checkpoints' / 'controlled'
ckpts = sorted(CKPT_DIR.glob('controlled_epoch_*.pt')) if CKPT_DIR.exists() else []
CKPT_PATH = ckpts[-1] if ckpts else None
if not CKPT_PATH:
    last = CKPT_DIR / 'controlled_last.pt' if CKPT_DIR.exists() else None
    CKPT_PATH = last if last and last.is_file() else None
print('Checkpoint:', CKPT_PATH)

In [ ]:
if CKPT_PATH:
    state = torch.load(CKPT_PATH, map_location='cpu')
    print('Checkpoint keys:', list(state.keys()))
    # Rebuild score model from saved config.
    model_cfg = ScoreUNetConfig(**state['score_config']) if 'score_config' in state else ScoreUNetConfig(in_channels=1, base_channels=32, channel_mults=(1,2,2))
    score_model = build_score_model(model_cfg)
    score_model.load_state_dict(state.get('score_state_dict', state.get('model')))
    score_model.to(DEVICE).eval()
    print('Score model loaded.')
    # Rebuild control policy from saved config.
    ctrl_cfg = ControlConfig(**state['control_config']) if 'control_config' in state else ControlConfig(state_dim=784, hidden_dim=128)
    control_policy = build_control_policy(ctrl_cfg, device=torch.device(DEVICE))
    control_policy.load_state_dict(state.get('control_state_dict', state.get('control')))
    control_policy.eval()
    print('Control policy loaded.')
else:
    print('No controlled checkpoint found. Run scripts/train_controlled_score.py first.')
    model_cfg = ScoreUNetConfig(in_channels=1, base_channels=32, channel_mults=(1,2,2))
    score_model = build_score_model(model_cfg).to(DEVICE).eval()
    ctrl_cfg = ControlConfig(state_dim=784, hidden_dim=64)
    control_policy = build_control_policy(ctrl_cfg, device=torch.device(DEVICE))
    print('Using uninitialised models — metrics will be meaningless.')

## 2. Controlled SDE sampling

In [ ]:
sde_cfg = ScoreSDEConfig(
    beta_min=0.1, beta_max=5.0, num_steps=50,
    sigma_min=0.01, sigma_max=1.0, control_weight=1.0,
    corrector_steps=1, corrector_step_size=0.01,
)
simulator = ScoreSDESimulator(score_model, sde_cfg)
BATCH = 16
with torch.no_grad():
    ctrl_samples, ctrl_stats = simulator.sample(
        shape=(BATCH, model_cfg.in_channels, 28, 28),
        device=torch.device(DEVICE),
        control=control_policy,
        return_stats=True,
    )
print('Shape:', ctrl_samples.shape)
print('Stats:', ctrl_stats)

In [ ]:
import matplotlib.pyplot as plt
from torchvision.utils import make_grid

grid = make_grid(ctrl_samples.cpu().clamp(-1, 1) * 0.5 + 0.5, nrow=4)
plt.figure(figsize=(8, 8))
plt.imshow(grid.permute(1, 2, 0).squeeze().numpy(), cmap='gray' if model_cfg.in_channels == 1 else None)
plt.axis('off')
plt.title('Controlled SDE samples')
plt.show()

## 3. Thermodynamic diagnostics

In [ ]:
from its.physics.entropy import entropy_production_estimate
from its.physics.fluctuation import jarzynski_work_estimate
import numpy as np

if 'control_energy' in ctrl_stats:
    print(f"Control energy:  {ctrl_stats['control_energy']:.4f}")
if 'path_kl' in ctrl_stats:
    print(f"Path KL:         {ctrl_stats['path_kl']:.4f}")
if 'jarzynski' in ctrl_stats:
    print(f"Jarzynski dF:    {ctrl_stats['jarzynski']:.4f}")

## 4. Quick benchmark (256 samples)

In [ ]:
from its.eval import EvaluationConfig, evaluate_sampler
from its.eval.evaluator import evaluate_ddpm_baseline

eval_cfg = EvaluationConfig(dataset_name='fashionmnist', num_samples=256, batch_size=64, device=DEVICE)

print('Evaluating controlled SDE ...')
ctrl_results = evaluate_sampler(score_model, control_policy, sde_cfg, eval_cfg)
print('Controlled SDE:', ctrl_results)

print('Evaluating DDPM ...')
ddpm_results = evaluate_ddpm_baseline(score_model, sde_cfg, eval_cfg, baseline='ddpm')
print('DDPM:', ddpm_results)

In [ ]:
# Side-by-side summary table
import pandas as pd

rows = [
    {'sampler': 'controlled_sde', **ctrl_results},
    {'sampler': 'ddpm', **ddpm_results},
]
df = pd.DataFrame(rows).set_index('sampler')
print(df.to_string())